In [9]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex, SearchField, SearchFieldDataType, SimpleField, SearchableField,
    VectorSearch, HnswAlgorithmConfiguration, VectorSearchProfile,
    AzureOpenAIVectorizer, AzureOpenAIVectorizerParameters
)

In [ ]:
# 1. Configure the Vectorizer (Automates query-time vectorization)
vectorizer = AzureOpenAIVectorizer(
    vectorizer_name="myVectorizer",
    parameters=AzureOpenAIVectorizerParameters(
        resource_url="https://temp-m2web-ai-search-pr-resource.cognitiveservices.azure.com/",
        deployment_name="text-embedding-3-small",
        model_name="text-embedding-3-small",
        api_key="<REDACTED_API_KEY>"
    )
)

In [11]:
# 2. Configure the Profile (Links Algorithm and Vectorizer)
vector_search = VectorSearch(
    algorithms=[HnswAlgorithmConfiguration(name="myHnsw")],
    profiles=[VectorSearchProfile(name="myProfile", algorithm_configuration_name="myHnsw", vectorizer_name="myVectorizer")],
    vectorizers=[vectorizer]
)

In [12]:
# 3. Define the Fields
fields = [
    SimpleField(name="id", type=SearchFieldDataType.String, key=True),
    SearchableField(name="content", type=SearchFieldDataType.String),
    SearchField(
        name="vector", 
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        vector_search_dimensions=1536,
        vector_search_profile_name="myProfile"
    )
]

In [ ]:
# 4. Initialize Client and Create Index
index = SearchIndex(name="integrated-index", fields=fields, vector_search=vector_search)
index_client = SearchIndexClient(
    endpoint="https://rush-lyric-ai-search.search.windows.net", 
    credential=AzureKeyCredential("<REDACTED_API_KEY>")
)

index_client.create_index(index)